# 02 · Content-Based Filtering

**Role in the pipeline:** base scorer #2. Produces `scores_cbf_{stack_train,val,test}` in the shared output format.

**Training-free by design.** Last year's CBF was *content similarities + a logistic regression on top*. Here CBF is a pure scorer and the only trained model is the stacker in `04_stack`, which keeps the comparison clean.

| Baseline limitation | Fix here |
|---|---|
| 50-topic TF-IDF vectors | 384-d **MiniLM** `article_content_emb` (group feature) on title + imputed abstract |
| Static profile: flat average of all past clicks | **Time-decayed** profile `Σ exp(-age/τ)·emb`, τ tuned on validation |
| Binary category match only | Continuous **category affinity** of the user for the candidate's category |
| Unreliable recency feature (first impression ≠ publication) | Dropped – banned by the model contract |
| Users with no history get a zero profile and a meaningless score | Marked `available = 0` → train-mean imputation + flag (contract) |

**Score** = `cos(user_profile, article_emb) + β · category_affinity + γ · cos(user_kg_profile, kg_entity_emb)` with β, γ tuned on validation. All three components are also saved as separate columns for the stacker.

In [8]:
import os
import json
import time
import numpy as np
import pandas as pd
from scipy import sparse

import common as C
import importlib
importlib.reload(C)

beh = C.load_behaviors()
cand = C.load_candidates(beh)
news = C.load_news()
nidx = pd.Series(news["nidx"].to_numpy(), index=news["news_id"])
print(f"articles: {len(news):,} | candidate rows: {len(cand):,}")

articles: 65,238 | candidate rows: 8,584,442


## Article representations
`article_content_emb`: `sentence-transformers/all-MiniLM-L6-v2`, mean pooling, max 64 tokens (same recipe as `02_feature_engineering`), L2-normalised and cached. If `transformers` is unavailable it falls back to TF-IDF + SVD so the notebook still runs (with a warning).

`kg_entity_emb`: mean of 100-d TransE vectors of the title + abstract Wikidata entities (zero vector if none).

In [9]:
def content_embeddings(news, batch_size=256):
    cache = os.path.join(C.CACHE_DIR, "article_content_emb_minilm.npy")
    if os.path.exists(cache):
        return np.load(cache)
    texts = (news["title"] + " " + news["abstract_imputed"].astype(str)).tolist()
    try:
        import torch
        from transformers import AutoTokenizer, AutoModel
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tok = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
        mdl = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").to(device).eval()
        out = []
        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                enc = tok(texts[i:i + batch_size], padding=True, truncation=True,
                          max_length=64, return_tensors="pt").to(device)
                h = mdl(**enc).last_hidden_state
                m = enc["attention_mask"].unsqueeze(-1).float()
                out.append(((h * m).sum(1) / m.sum(1).clamp(min=1e-9)).cpu().numpy())
        E = np.vstack(out).astype(np.float32)
    except Exception as e:  # noqa: BLE001
        print(f"WARNING: MiniLM unavailable ({type(e).__name__}: {e}). Falling back to TF-IDF+SVD.")
        cache = os.path.join(C.CACHE_DIR, "article_content_emb_tfidf.npy")
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD
        X = TfidfVectorizer(min_df=2, max_features=50000).fit_transform(texts)
        E = TruncatedSVD(n_components=min(384, X.shape[1] - 1), random_state=C.SEED).fit_transform(X).astype(np.float32)
    E /= np.maximum(np.linalg.norm(E, axis=1, keepdims=True), 1e-12)
    np.save(cache, E)
    return E


def kg_embeddings(news, data_dir=C.DATA_DIR):
    vecs = {}
    for s in ("train", "dev"):
        p = f"{data_dir}/{s}/entity_embedding.vec"
        if os.path.exists(p):
            with open(p) as f:
                for line in f:
                    parts = line.rstrip("\n").split("\t")
                    if len(parts) >= 101 and parts[0] not in vecs:
                        vecs[parts[0]] = np.asarray(parts[1:101], dtype=np.float32)
    K = np.zeros((len(news), 100), np.float32)
    for i, (t, a) in enumerate(zip(news["title_entities"], news["abstract_entities"])):
        ids = []
        for js in (t, a):
            try:
                ids += [e.get("WikidataId") for e in json.loads(js)]
            except (TypeError, ValueError):
                pass
        v = [vecs[q] for q in ids if q in vecs]
        if v:
            K[i] = np.mean(v, axis=0)
    K /= np.maximum(np.linalg.norm(K, axis=1, keepdims=True), 1e-12)
    return K


t0 = time.time()
E = content_embeddings(news)
K = kg_embeddings(news)
cat_codes, cat_names = pd.factorize(news["category"])
print(f"content emb {E.shape}, kg emb {K.shape} ({(np.abs(K).sum(1) > 0).mean():.1%} with entities), "
      f"{len(cat_names)} categories, {time.time() - t0:.0f}s")

content emb (65238, 384), kg emb (65238, 100) (86.1% with entities), 18 categories, 1s


## User profiles and candidate features

In [10]:
def user_profiles(qlog, tau_days):
    """Time-decayed content, KG and category profiles for every user in qlog."""
    q = qlog[qlog["news_id"].isin(nidx.index)]
    users = pd.Index(q["user_id"].unique())
    w = C.decay_weights(q["time"], q["time"].max(), tau_days)
    W = sparse.csr_matrix((w, (users.get_indexer(q["user_id"]), nidx[q["news_id"]].to_numpy())),
                          shape=(len(users), len(news)))
    P = np.asarray(W @ E)
    P /= np.maximum(np.linalg.norm(P, axis=1, keepdims=True), 1e-12)
    PK = np.asarray(W @ K)
    PK /= np.maximum(np.linalg.norm(PK, axis=1, keepdims=True), 1e-12)
    onehot = sparse.csr_matrix((np.ones(len(news), np.float32), (np.arange(len(news)), cat_codes)),
                               shape=(len(news), len(cat_names)))
    A = np.asarray((W @ onehot).todense())
    A /= np.maximum(A.sum(axis=1, keepdims=True), 1e-12)
    return users, P, PK, A


def candidate_features(c, qlog, tau_days, chunk=500_000):
    users, P, PK, A = user_profiles(qlog, tau_days)
    u = users.get_indexer(c["user_id"])
    a = nidx.reindex(c["news_id"]).fillna(-1).astype(int).to_numpy()
    ok = (u >= 0) & (a >= 0)
    content, kg, aff = (np.full(len(c), np.nan, np.float32) for _ in range(3))
    idx = np.flatnonzero(ok)
    for s in range(0, len(idx), chunk):
        j = idx[s:s + chunk]
        content[j] = np.einsum("ij,ij->i", P[u[j]], E[a[j]])
        kg[j] = np.einsum("ij,ij->i", PK[u[j]], K[a[j]])
        aff[j] = A[u[j], cat_codes[a[j]]]
    return pd.DataFrame(index=c.index, data={
        "cbf_content": content, "cbf_kg": kg, "cbf_cat_aff": aff,
        "available": ok.astype(np.int8)})


def phase_features(phase, tau_days):
    cfg = C.PHASES[phase]
    log = C.build_click_log(beh, cand, cfg["build_days"])
    c = C.phase_candidates(cand, phase)
    c = C.add_cold_start_flags(c, beh, log, cfg["score_days"])
    qlog = C.user_query_log(log, beh, cfg["score_days"], c["user_id"])
    return pd.concat([c, candidate_features(c, qlog, tau_days)], axis=1)


def combine(df, beta, gamma):
    return df["cbf_content"] + beta * df["cbf_cat_aff"] + gamma * df["cbf_kg"]

## Tune τ (time decay), β (category affinity), γ (KG) on validation

In [11]:
runs = []
for tau in (None, 1.0, 3.0):
    v = phase_features("val", tau)
    for beta in (0.0, 0.25, 0.5, 1.0):
        for gamma in (0.0, 0.25):
            s = combine(v, beta, gamma)
            vv = v.assign(score=s.fillna(s.mean()))
            r = C.evaluate(vv, by_segment=False).iloc[0]
            runs.append({"tau_days": tau, "beta": beta, "gamma": gamma, **r.to_dict()})
tuning = pd.DataFrame(runs).sort_values("nDCG@10", ascending=False)
best = tuning.iloc[0]
BEST = {"tau_days": None if pd.isna(best["tau_days"]) else float(best["tau_days"]),
        "beta": float(best["beta"]), "gamma": float(best["gamma"])}
print("best:", BEST)
tuning.head(10).round(4)

best: {'tau_days': None, 'beta': 0.25, 'gamma': 0.0}


,tau_days,beta,gamma,impressions,MRR,nDCG@5,nDCG@10,AUC
2,NaN,0.25,0.00,30270.0,0.3448,0.3207,0.3798,0.6428
0,NaN,0.00,0.00,30270.0,0.3412,0.3177,0.3773,0.6404
4,NaN,0.50,0.00,30270.0,0.3425,0.3175,0.3767,0.6393
18,3.0,0.25,0.00,30270.0,0.3408,0.3175,0.3758,0.6391
3,NaN,0.25,0.25,30270.0,0.3408,0.3177,0.3754,0.6396
1,NaN,0.00,0.25,30270.0,0.3399,0.3166,0.3740,0.6378
5,NaN,0.50,0.25,30270.0,0.3387,0.3156,0.3734,0.6377
19,3.0,0.25,0.25,30270.0,0.3391,0.3151,0.3734,0.6367
16,3.0,0.00,0.00,30270.0,0.3377,0.3144,0.3733,0.6366
20,3.0,0.50,0.00,30270.0,0.3385,0.3139,0.3729,0.6360


## Score all phases and save (shared output format)

In [12]:
EXTRA = ["cbf_content", "cbf_kg", "cbf_cat_aff"]
for phase in C.PHASES:
    df = phase_features(phase, BEST["tau_days"])
    df["score"] = combine(df, BEST["beta"], BEST["gamma"])
    path = C.save_scores(df, "cbf", phase, extra_cols=EXTRA)
    print(f"{phase:12s} rows={len(df):,}  available={df['available'].mean():.1%} -> {path}")

stack_train  rows=1,231,673  available=99.4% -> /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4222/Project/bt4222-group12/models/2_UBCF + CBF/outputs/scores_cbf_stack_train.parquet
val          rows=1,222,429  available=99.5% -> /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4222/Project/bt4222-group12/models/2_UBCF + CBF/outputs/scores_cbf_val.parquet
test         rows=2,740,998  available=97.3% -> /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4222/Project/bt4222-group12/models/2_UBCF + CBF/outputs/scores_cbf_test.parquet


## Standalone evaluation (val & test, by cold-start segment)

In [13]:
res = C.standalone_eval("cbf")
for ph, table in res.items():
    table = table[["impressions", "MRR", "nDCG@5", "nDCG@10", "AUC"]]
    print(f"\n=== CBF · {ph} ===")
    print(table.round(4).to_string())
    table.to_csv(f"{C.OUT_DIR}/metrics_cbf_{ph}.csv")


=== CBF · val ===
                                       impressions     MRR  nDCG@5  nDCG@10     AUC
All                                          30270  0.3448  0.3207   0.3798  0.6428
Warm users                                   30110  0.3454  0.3212   0.3803  0.6436
Cold users (0 clicks)                          160  0.2455  0.2298   0.2806  0.5000
Clicked article seen before                   8276  0.2776  0.2598   0.3154  0.6251
Clicked article new (no prior clicks)        21994  0.3701  0.3437   0.4040  0.6494

=== CBF · test ===
                                       impressions     MRR  nDCG@5  nDCG@10     AUC
All                                          73152  0.3622  0.3452   0.4036  0.6448
Warm users                                   71152  0.3651  0.3483   0.4066  0.6488
Cold users (0 clicks)                         2000  0.2565  0.2347   0.2954  0.5000
Clicked article seen before                   8137  0.3982  0.4277   0.4634  0.5275
Clicked article new (no prior clicks)